# Chapter 5 — GEODE: self-correcting extraction

Chapter 4 built a store from the annual report by trusting the extractor.
This chapter does not trust it. Before a triple is indexed, GEODE runs a
closed loop that lets the **geometry judge the extraction**: a composition
critic catches relational triples that contradict the rest of the graph, and
external **anchors** (a declared sum, a duplicate value) catch numeric errors
the geometry is blind to. We corrupt the corpus on purpose and watch the loop
flag each break **with provenance** — then state, honestly, the one class of
error neither can catch.

All symbols come from `knowlytix.knowledge.geode`. The actor LLM, when used,
is Qwen 3B only (research-integrity lock) — Claude is never in the loop.

In [ ]:
import os, sys
# Locate this topic's code/ dir (book_kit.py), robust to the working directory.
_cwd = os.getcwd()
for _p in (_cwd, os.path.join(os.path.dirname(_cwd), "code"), os.path.join(_cwd, "code")):
    if os.path.isfile(os.path.join(_p, "book_kit.py")):
        break
else:
    _p = os.environ.get("GMS_RAG_TUTORIAL", _cwd)
sys.path.insert(0, _p)
from book_kit import CORPUS, DEVICE  # repo-root paths + GMS-knowlytix bootstrap


## The triples, before correction

We work over the canonical Northwind FY2025 triples (see `data/corpus_facts.md`).
The segment table declares a `Total` row, and `Total` revenue equals the sum of
the four segment revenues — `120 + 80 + 95 + 60 == 355`. That redundancy is what
the **sum anchor** checks. The `segment -> division -> region` chain is the
redundancy the **composition critic** checks.

In [ ]:
# Canonical triples from the annual report (grounded in data/corpus_facts.md).
# In a real run these come from `ingest_markdown` / `build_rag_store`; here we
# state them explicitly so the corruption is unambiguous.
clean_triples = [
    ("cloud platform", "has_revenue", "120.0"),
    ("devices", "has_revenue", "80.0"),
    ("logistics", "has_revenue", "95.0"),
    ("retail", "has_revenue", "60.0"),
    ("total", "has_revenue", "355.0"),
    ("cloud platform", "has_division", "technology"),
    ("devices", "has_division", "technology"),
    ("logistics", "has_division", "operations"),
    ("retail", "has_division", "operations"),
    ("technology", "has_region", "north america"),
    ("operations", "has_region", "europe"),
]

# The Total row declares: Total = sum of the four segments.
parts = [120.0, 80.0, 95.0, 60.0]
assert sum(parts) == 355.0

## Listing 1 — corrupt a segment figure: the sum anchor flags it

Change Devices revenue from `80.0` to `88.0`. Now `120 + 88 + 95 + 60 = 363`,
but the `Total` row still says `355.0`. No single triple is *internally* wrong —
the geometry cannot see it. The **sum anchor** can: it reads the exact values
back through the ENM register and compares the declared total to the sum of
parts. `AnchorChecker.auto_sum_constraints` derives the constraint from the
`Total` row the way a financial table already declares it.

In [ ]:
from knowlytix.knowledge.geode import (
    enm_from_triples, AnchorChecker, ProvenanceLedger,
)

# Inject the sum break: Devices 80.0 -> 88.0.
corrupted = [
    (h, r, "88.0") if (h, r) == ("devices", "has_revenue") else (h, r, t)
    for (h, r, t) in clean_triples
]

# Build the integrity-checked exact register, then the anchor checker.
enm = enm_from_triples(corrupted)
ledger = ProvenanceLedger.from_text(open(CORPUS).read(),
                                    CORPUS)
checker = AnchorChecker.from_enm(enm, ledger)

# The sum constraint is derived automatically from the "total" row.
constraints = checker.auto_sum_constraints()
for c in constraints:
    print("constraint:", c.total, "= sum", [p[0] for p in c.parts])

violations = checker.check_all(corrupted)
for v in violations:
    print(v.kind, "|", v.message)
    for loc in v.locations:
        print("   provenance:", loc.location(), "raw:", repr(loc.raw))

**Expected.** `auto_sum_constraints` derives `('total','has_revenue') = sum of
[cloud platform, devices, logistics, retail]`. `check_all` returns a `sum`
violation: declared `total.has_revenue=355` `!=` `sum(parts)=363` (off by `8`),
with a provenance location pointing back into the segment table. The anchor
**detects** the break and names the candidate parts; it does not auto-rewrite
a number — which part is wrong is a review decision (the honest localization
limit of a sum constraint).

## Listing 2 — the full closed loop over the real report

Now run the *whole* loop end to end over the real annual report with one
injected break. The loop ingests the document with regex, trains a small GMS,
runs the `CompositionCritic` to repair any *relational* contradictions, resolves
duplicate tails, and finally runs the **external anchors** for the numeric
errors the geometry is blind to. We inject a single segment-revenue break
(Devices `80.0` -> `88.0`) so the four segments sum to `363` while the `Total`
row still declares `355.0`. No single triple is internally wrong, so only the
**sum anchor** -- derived automatically from the `Total` row -- surfaces it.

This cell trains a small GMS, so it is **GPU/CI-only** -- do not run it here.

In [ ]:
from knowlytix.knowledge.geode import GeodeLoop, make_default_trainer

# Run the GEODE loop on a copy of the REAL annual report (data/annual_report.md)
# with ONE injected corruption: Devices revenue 80.0 -> 88.0. The four segments
# now sum to 120 + 88 + 95 + 60 = 363, but the Total row still declares 355.0.
# No single triple is internally wrong (the geometry is blind to it); the SUM
# ANCHOR, derived automatically from the Total row, catches the break.
#
# We use the full report (not a 2-row stub) so the regex ingest produces enough
# triples to form a real training batch -- the loop trains a small GMS, so this
# is GPU/CI-only; do not run it on a shared box without a free GPU.
import tempfile

source = open(CORPUS).read()
# Inject exactly one sum break in the Segment Performance table.
corrupted_doc = source.replace(
    "| Devices | Technology | 80.0 | 210 |",
    "| Devices | Technology | 88.0 | 210 |",
)
assert corrupted_doc != source, "expected to inject one Devices-revenue break"

tmp = tempfile.NamedTemporaryFile("w", suffix=".md", delete=False)
tmp.write(corrupted_doc)
tmp.close()

# Geometry is authoritative for relational repairs; the sum anchor (check_anchors
# on, the default) handles the numeric break the geometry cannot see. The trainer
# is injected so orchestration is testable; this still trains a GMS -> GPU/CI only.
loop = GeodeLoop(make_default_trainer(DEVICE, epochs=40))
result = loop.run(tmp.name)

print("converged:", result.converged, "| iterations:", result.iterations)
for v in result.anchor_violations:
    print("anchor:", v["kind"], "|", v["message"], "| residual:", round(v["residual"], 3))

# Self-check (runs only on the GPU/CI path): the injected sum break surfaces as
# a "sum" anchor violation in the loop's result. Grounded in the corpus facts --
# 120 + 88 + 95 + 60 = 363 vs the declared Total 355.0, residual 8.0.
sum_breaks = [v for v in result.anchor_violations if v["kind"] == "sum"]
assert sum_breaks, "injected sum break must appear in result.anchor_violations"
assert abs(sum_breaks[0]["residual"] - 8.0) < 1e-6, sum_breaks[0]["residual"]
print("OK -- loop's anchor layer flagged the injected sum break:", sum_breaks[0]["message"])


**Expected.** The composition critic finds no relational contradiction (the
segment->division->region chain is intact), so the loop converges and runs the
external anchors. `result.anchor_violations` contains a `sum` violation:
declared `total.has_revenue=355` `!=` `sum(parts)=363` (residual `8`), localized
to the segment table. Unlike a relational error -- which the geometry can
auto-fix because the graph carries a second source of truth (the composition
path) -- a lone corrupted number has no redundant cross-check, so the sum anchor
**detects and localizes** the break but leaves which part is wrong to review.
This is the honest division of labour: geometry repairs relational
contradictions; declared anchors catch the numeric breaks geometry cannot see.

## Listing 3 — the honest limit: a lone value with no redundancy

Consider `total_assets has_amount 540.0` (Balance Sheet). It appears once, with
no `Total = Σ parts` row over the balance-sheet line items and no second
statement of the figure. Corrupt it to `999.0` and **neither** mechanism fires:
the composition critic needs a redundant relational chain (there is none), and
the sum anchor needs declared parts (there are none). This is the boundary the
design doc states plainly.

In [ ]:
# A lone numeric fact: one value, no parts, no duplicate, no composition chain.
lone = [("total assets", "has_amount", "999.0")]   # truth is 540.0

enm_lone = enm_from_triples(lone)
checker_lone = AnchorChecker.from_enm(enm_lone)

# No sum constraint can be derived (no "total" row over parts), and there is no
# duplicate of the same (entity, relation) -> nothing to cross-check.
print("derived sum constraints:", checker_lone.auto_sum_constraints())
print("anchor violations:", checker_lone.check_all(lone))

**Expected.** `auto_sum_constraints` returns `[]` and `check_all` returns `[]`:
the corrupted `999.0` passes silently. Geometry corrects what violates
redundancy; it cannot invent a cross-check that the document never provided.
The remedy is upstream — declare a constraint, add a duplicate statement, or
carry the figure as an ENM-checked exact value sourced to its one cell — not a
better critic.

## Exercise — the duplicate anchor

Assert the *same* `(entity, relation)` twice with conflicting exact values and
confirm the **duplicate anchor** fires (precise localization, unlike the sum
case). Worked solution below.

In [ ]:
# EXERCISE SOLUTION — conflicting duplicate of the same (entity, relation).
dup = clean_triples + [("total", "has_revenue", "350.0")]  # conflicts with 355.0

checker_dup = AnchorChecker.from_enm(enm_from_triples(dup))
dups = checker_dup.check_duplicates(dup)
for v in dups:
    print(v.kind, "|", v.message, "| residual:", v.residual)

assert any(d.kind == "duplicate" for d in dups), "duplicate anchor should fire"

## Self-check

The chapter's claim: when a segment figure is corrupted so `Total != Σ parts`,
the loop's anchor layer surfaces the break. We assert the injected sum break
appears in the anchor violations — exactly the property the brief's self-check
names (`anchor_violations` contains the injected sum break).

In [ ]:
# Self-check: the injected sum break (Devices 80 -> 88) is detected as a "sum"
# anchor violation. This is the CPU-only anchor path (no GMS training, no Qwen),
# so it runs deterministically in CI.
enm_chk = enm_from_triples(corrupted)
checker_chk = AnchorChecker.from_enm(enm_chk)
viols = checker_chk.check_all(corrupted)

sum_viols = [v for v in viols if v.kind == "sum"]
assert sum_viols, "expected a sum anchor violation for the corrupted segment"
v = sum_viols[0]
assert abs(v.residual - 8.0) < 1e-6, f"expected residual 8.0, got {v.residual}"
print("OK — sum anchor flagged the injected break:", v.message)